# Naan Binary — Ingredient-Based Recipe Generator
### Updated notebook: GPU training, checkpointing, repetition fix, more data

In [3]:
# ── Cell 1: Imports & seed ────────────────────────────────────────────────────
import importlib, subprocess, sys

required_pkgs = ["transformers", "accelerate", "sentencepiece", "isodate", "scikit-learn"]
missing = [p for p in required_pkgs
           if not importlib.util.find_spec("sklearn" if p == "scikit-learn" else p)]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import ast, csv, os, re
from io import StringIO
from typing import List

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import isodate

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── GPU check ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA not available — training will be very slow on CPU!")

Using device : cuda
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM         : 8.6 GB


In [4]:
# ── Cell 2: Load dataset ──────────────────────────────────────────────────────
df = pd.read_csv("cleaned_recipes.csv")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.info()

Loaded 157,973 rows, 22 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157973 entries, 0 to 157972
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RecipeId                    157973 non-null  int64  
 1   Name                        157973 non-null  object 
 2   CookTime                    157973 non-null  object 
 3   PrepTime                    157973 non-null  object 
 4   TotalTime                   157973 non-null  object 
 5   Description                 157973 non-null  object 
 6   Images                      157973 non-null  object 
 7   RecipeCategory              157973 non-null  object 
 8   Keywords                    157973 non-null  object 
 9   RecipeIngredientQuantities  157973 non-null  object 
 10  RecipeIngredientParts       157973 non-null  object 
 11  Calories                    157973 non-null  float64
 12  FatContent                  157973 non-n

In [5]:
# ── Cell 3: Preprocessing ─────────────────────────────────────────────────────
NUTRITION_COLS = [
    "Calories", "FatContent", "SaturatedFatContent", "CholesterolContent",
    "SodiumContent", "CarbohydrateContent", "FiberContent", "SugarContent", "ProteinContent",
]

UNIT_WORDS = {
    "tsp", "teaspoon", "teaspoons", "tbsp", "tablespoon", "tablespoons",
    "cup", "cups", "oz", "ounce", "ounces", "lb", "lbs", "pound", "pounds",
    "g", "gram", "grams", "kg", "ml", "l", "liter", "liters", "pinch", "dash",
    "clove", "cloves", "slice", "slices", "can", "cans", "package", "packages",
}

def parse_list_like(value) -> List[str]:
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
    if text.startswith("c(") and text.endswith(")"):
        inner = text[2:-1].strip()
        reader = csv.reader(StringIO(inner), delimiter=",", quotechar='"', skipinitialspace=True)
        row = next(reader, [])
        out = []
        for token in row:
            token = token.strip().replace('""', '"')
            if token in {"", "NA", "None", "nan"}:
                continue
            out.append(token)
        return out
    return [text]

def parse_iso_minutes(value) -> float:
    if pd.isna(value):
        return 0.0
    text = str(value).strip()
    if not text:
        return 0.0
    try:
        duration = isodate.parse_duration(text)
        if hasattr(duration, "total_seconds"):
            return float(duration.total_seconds() / 60.0)
    except Exception:
        pass
    m = re.match(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", text)
    if not m:
        return 0.0
    days = int(m.group(1) or 0)
    hours = int(m.group(2) or 0)
    minutes = int(m.group(3) or 0)
    return float(days * 24 * 60 + hours * 60 + minutes)

def normalize_ingredient(token: str) -> str:
    s = token.lower().strip()
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"^\s*(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s*", "", s)
    s = re.sub(r"^\s*(?:x|about|approx\.?|approximately)\s+", "", s)
    parts = s.split()
    while parts and re.fullmatch(r"\d+(?:\.\d+)?", parts[0]):
        parts.pop(0)
    while parts and parts[0] in UNIT_WORDS:
        parts.pop(0)
    s = " ".join(parts)
    s = re.sub(r"[^a-zA-Z\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip(" -")
    return s

def build_input_prompt(ingredients: List[str], total_minutes: float) -> str:
    cleaned = [normalize_ingredient(x) for x in ingredients]
    cleaned = [x for x in cleaned if x]
    cleaned = list(dict.fromkeys(cleaned))
    if len(cleaned) > 40:
        cleaned = cleaned[:40]
    prompt = f"ingredients: {', '.join(cleaned)}"
    if total_minutes > 0:
        prompt += f" | ready_in_min: {int(total_minutes)}"
    return prompt

def build_target_text(name: str, instructions: List[str]) -> str:
    steps = [s.strip() for s in instructions if s and s.strip()]
    if len(steps) > 25:
        steps = steps[:25]
    body = "\n".join(f"{i+1}. {step}" for i, step in enumerate(steps))
    title = str(name).strip() if pd.notna(name) else "Unknown Recipe"
    return f"name: {title}\nsteps:\n{body}"

# Build working dataframe
df_work = df.copy()
df_work["_ingredient_list"] = df_work["RecipeIngredientParts"].apply(parse_list_like)
df_work["_instruction_list"] = df_work["RecipeInstructions"].apply(parse_list_like)
df_work["_total_minutes"] = df_work["TotalTime"].apply(parse_iso_minutes)

# Fix servings
df_work["RecipeServings"] = pd.to_numeric(df_work["RecipeServings"], errors="coerce")
serving_median = df_work["RecipeServings"].dropna().median()
if pd.isna(serving_median) or serving_median <= 0:
    serving_median = 4.0
df_work["RecipeServings"] = df_work["RecipeServings"].fillna(serving_median).clip(lower=1.0)

# Normalize nutrition per serving
for col in NUTRITION_COLS:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")
    df_work[col] = df_work[col] / df_work["RecipeServings"]

# Build text columns
df_work["input_text"] = df_work.apply(
    lambda r: build_input_prompt(r["_ingredient_list"], r["_total_minutes"]), axis=1)
df_work["target_text"] = df_work.apply(
    lambda r: build_target_text(r["Name"], r["_instruction_list"]), axis=1)

# Filter bad rows
df_work = df_work[df_work["input_text"].str.len() > 15]
df_work = df_work[df_work["target_text"].str.len() > 20]
df_work = df_work.dropna(subset=NUTRITION_COLS).reset_index(drop=True)

print(f"Prepared rows: {len(df_work):,}")
print(df_work[["input_text", "target_text"]].head(2))

Prepared rows: 157,973
                                          input_text  \
0  ingredients: blueberries, granulated sugar, va...   
1  ingredients: saffron, milk, hot green chili pe...   

                                         target_text  
0  name: Low-Fat Berry Blue Frozen Dessert\nsteps...  
1  name: Biryani\nsteps:\n1. Soak saffron in warm...  


In [6]:
# ── Cell 4: Config — tweak these values if needed ─────────────────────────────

# Model — flan-t5-base is much better than small, fits 8GB VRAM comfortably
MODEL_NAME     = "google/flan-t5-base"   # upgrade to flan-t5-large if you want even better results
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 300

# Data — using full dataset now (was 30k before)
# MAX_ROWS       = len(df_work)            # use ALL 157k rows; set to e.g. 50000 for quick test runs
MAX_ROWS       = 20000            # use subset of data for quicker testing

# Paths
CKPT_DIR       = "./recipe_checkpoints"  # checkpoints saved here during training
MODEL_SAVE_DIR = "./recipe_model_final"  # final model saved here after training
SCALER_PATH    = "./recipe_scaler.pkl"   # scaler saved separately

print(f"Model        : {MODEL_NAME}")
print(f"Max rows     : {MAX_ROWS:,}")
print(f"Checkpoint   : {CKPT_DIR}")
print(f"Final save   : {MODEL_SAVE_DIR}")

Model        : google/flan-t5-base
Max rows     : 20,000
Checkpoint   : ./recipe_checkpoints
Final save   : ./recipe_model_final


In [7]:
# ── Cell 5: Train / val split & scaler ───────────────────────────────────────
if len(df_work) > MAX_ROWS:
    df_model = df_work.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
else:
    df_model = df_work.copy()

idx = np.arange(len(df_model))
train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=SEED)

train_df = df_model.iloc[train_idx].reset_index(drop=True)
val_df   = df_model.iloc[val_idx].reset_index(drop=True)

scaler = StandardScaler()
train_targets = scaler.fit_transform(train_df[NUTRITION_COLS].to_numpy(dtype=np.float32))
val_targets   = scaler.transform(val_df[NUTRITION_COLS].to_numpy(dtype=np.float32))

# Save scaler immediately so it's never lost
joblib.dump(scaler, SCALER_PATH)
print(f"Scaler saved to {SCALER_PATH}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

Scaler saved to ./recipe_scaler.pkl
Train: 18,000 | Val: 2,000


In [8]:
# ── Cell 6: Tokenizer & Dataset ───────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RecipeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, y_scaled: np.ndarray):
        self.frame = frame
        self.y = y_scaled

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        model_in = tokenizer(
            row["input_text"],
            truncation=True,
            max_length=MAX_INPUT_LEN,
        )
        model_out = tokenizer(
            text_target=row["target_text"],
            truncation=True,
            max_length=MAX_TARGET_LEN,
        )
        return {
            "input_ids":      model_in["input_ids"],
            "attention_mask": model_in["attention_mask"],
            "labels":         model_out["input_ids"],
            "reg_targets":    torch.tensor(self.y[idx], dtype=torch.float32),
        }

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=MODEL_NAME, padding=True)

def multitask_collate(features):
    reg_targets = torch.stack([f["reg_targets"] for f in features])
    core = [{k: v for k, v in f.items() if k != "reg_targets"} for f in features]
    batch = data_collator(core)
    batch["labels"][batch["labels"] == tokenizer.pad_token_id] = -100
    batch["reg_targets"] = reg_targets
    return batch

train_ds = RecipeDataset(train_df, train_targets)
val_ds   = RecipeDataset(val_df,   val_targets)

print(f"Train dataset: {len(train_ds):,} samples")
print(f"Val dataset  : {len(val_ds):,} samples")

Train dataset: 18,000 samples
Val dataset  : 2,000 samples


In [9]:
# ── Cell 7: Model definition ──────────────────────────────────────────────────
class MultiTaskFlanT5(nn.Module):
    def __init__(self, model_name: str, num_targets: int, reg_weight: float = 0.3):
        super().__init__()
        self.backbone   = T5ForConditionalGeneration.from_pretrained(model_name)
        hidden          = self.backbone.config.d_model
        # Deeper regression head for better nutrition prediction
        self.reg_head   = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_targets),
        )
        self.reg_weight = reg_weight

    def forward(self, input_ids=None, attention_mask=None, labels=None, reg_targets=None, **kwargs):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True,
        )
        # Mean pool encoder hidden states (weighted by attention mask)
        enc_hidden = out.encoder_last_hidden_state
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (enc_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        reg_pred = self.reg_head(pooled)

        loss     = out.loss
        reg_loss = None
        if reg_targets is not None:
            reg_loss = nn.functional.mse_loss(reg_pred, reg_targets)
            loss     = loss + self.reg_weight * reg_loss

        return {
            "loss":      loss,
            "logits":    out.logits,
            "reg_logits": reg_pred,
            "ce_loss":   out.loss.detach()   if out.loss  is not None else None,
            "reg_loss":  reg_loss.detach()   if reg_loss  is not None else None,
        }

    @torch.no_grad()
    def generate_text(self, input_ids, attention_mask, **kwargs):
        return self.backbone.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

model = MultiTaskFlanT5(MODEL_NAME, num_targets=len(NUTRITION_COLS), reg_weight=0.3)
print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: google/flan-t5-base
Trainable params: 247,808,777


In [10]:
# ── Cell 8: Training ──────────────────────────────────────────────────────────
# These settings are tuned for your RTX 4060 (8GB VRAM)
# If you get CUDA out of memory: set per_device_train_batch_size=8, gradient_accumulation_steps=2

training_args = TrainingArguments(
    output_dir=CKPT_DIR,

    # ── How long to train ──────────────────────────────────────────────────────
    num_train_epochs=3,                  # 3 full passes over the data

    # ── Batch size (GPU memory) ────────────────────────────────────────────────
    per_device_train_batch_size=16,      # reduce to 8 if OOM
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,       # increase to 2 if batch_size reduced to 8

    # ── Speed optimisations ────────────────────────────────────────────────────
    fp16=torch.cuda.is_available(),      # float16 on GPU — ~2x faster, half VRAM
    dataloader_num_workers=0,            # parallel data loading
    dataloader_pin_memory=False,          # faster CPU→GPU transfer

    # ── Learning rate ─────────────────────────────────────────────────────────
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,                   # 5% of steps for LR warmup
    lr_scheduler_type="cosine",          # smoother decay than linear

    # ── Checkpointing — model is saved here automatically ─────────────────────
    save_strategy="epoch",               # save at end of every epoch
    save_total_limit=2,                  # keep only the 2 most recent checkpoints
    load_best_model_at_end=True,         # auto-load best checkpoint when done
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── Evaluation ────────────────────────────────────────────────────────────
    eval_strategy="epoch",               # evaluate once per epoch

    # ── Logging ───────────────────────────────────────────────────────────────
    logging_steps=100,
    logging_dir="./logs",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=multitask_collate,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop if val loss stops improving
)

# ── Resume from checkpoint if one exists (safe to re-run cell) ────────────────
last_ckpt = None
if os.path.isdir(CKPT_DIR):
    ckpts = [os.path.join(CKPT_DIR, d) for d in os.listdir(CKPT_DIR)
             if d.startswith("checkpoint-")]
    if ckpts:
        last_ckpt = max(ckpts, key=os.path.getmtime)
        print(f"Resuming from checkpoint: {last_ckpt}")
    else:
        print("No checkpoint found — starting fresh training")
else:
    print("No checkpoint dir found — starting fresh training")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)
print("\nTraining complete!")
print(train_result.metrics)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


No checkpoint found — starting fresh training


Epoch,Training Loss,Validation Loss
1,0.000000,nan


RuntimeError: 
            Some tensors share memory, this will lead to duplicate memory on disk and potential differences when loading them again: [{'backbone.decoder.embed_tokens.weight', 'backbone.shared.weight', 'backbone.encoder.embed_tokens.weight'}].
            A potential way to correctly save your model is to use `save_model`.
            More information at https://huggingface.co/docs/safetensors/torch_shared_tensors
            

In [ ]:
# ── Cell 9: Save final model ──────────────────────────────────────────────────
# Run this once after training is done.
# After this, you NEVER need to retrain — just load from MODEL_SAVE_DIR.

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Save the T5 backbone (handles generation)
model.backbone.save_pretrained(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)

# Save the regression head weights separately
torch.save(model.reg_head.state_dict(), os.path.join(MODEL_SAVE_DIR, "reg_head.pt"))

# Scaler was already saved in Cell 5, but save again to be safe
joblib.dump(scaler, os.path.join(MODEL_SAVE_DIR, "scaler.pkl"))

print(f"Model saved to   : {MODEL_SAVE_DIR}/")
print(f"Contents         : {os.listdir(MODEL_SAVE_DIR)}")

In [ ]:
# ── Cell 10: Load saved model (run this instead of training next time) ─────────
# Run only this cell on future sessions — no retraining needed!

tokenizer  = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)
scaler     = joblib.load(os.path.join(MODEL_SAVE_DIR, "scaler.pkl"))

model      = MultiTaskFlanT5(MODEL_SAVE_DIR, num_targets=len(NUTRITION_COLS))
model.reg_head.load_state_dict(
    torch.load(os.path.join(MODEL_SAVE_DIR, "reg_head.pt"), map_location=DEVICE)
)
model.to(DEVICE)
model.eval()

print("Model loaded from disk and ready for inference!")

In [ ]:
# ── Cell 11: Inference ────────────────────────────────────────────────────────
def generate_recipe(ingredient_input: str, max_length: int = MAX_TARGET_LEN):
    """
    ingredient_input: plain comma-separated string
    e.g. "chicken breast, garlic, lemon, olive oil, rosemary"
    """
    # Build the prompt the same way training data was built
    raw_ingredients = [x.strip() for x in ingredient_input.split(",")]
    prompt = build_input_prompt(raw_ingredients, total_minutes=0)

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        # ── Recipe generation (repetition fixes applied here) ─────────────────
        gen_ids = model.generate_text(
            enc["input_ids"],
            enc["attention_mask"],
            max_length=max_length,
            num_beams=4,
            no_repeat_ngram_size=3,   # prevents 3-grams from repeating
            repetition_penalty=1.3,   # penalises reusing already-generated tokens
            early_stopping=True,
        )

        # ── Nutrition prediction ──────────────────────────────────────────────
        enc_hidden = model.backbone.encoder(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            return_dict=True,
        ).last_hidden_state
        mask   = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (enc_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        pred_scaled = model.reg_head(pooled).cpu().numpy()

    recipe    = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    nutrition = scaler.inverse_transform(pred_scaled)[0]

    print("=" * 60)
    print(f"INPUT  : {prompt}")
    print("=" * 60)
    print("RECIPE :")
    print(recipe)
    print("\nNUTRITION (per serving):")
    for col, val in zip(NUTRITION_COLS, nutrition):
        print(f"  {col:<25}: {val:.2f}")
    print("=" * 60)
    return recipe, nutrition

# ── Example usage ─────────────────────────────────────────────────────────────
generate_recipe("chicken breast, garlic, lemon, olive oil, rosemary")

In [ ]:
# ── Cell 12: Evaluation on validation set ─────────────────────────────────────
# Computes MAE for each nutrition column to measure regression quality

from sklearn.metrics import mean_absolute_error

model.eval()
all_preds, all_true = [], []

# Use a small subset for quick evaluation
eval_subset = val_df.head(200)
eval_targets = scaler.transform(eval_subset[NUTRITION_COLS].to_numpy(dtype=np.float32))

with torch.no_grad():
    for i in range(len(eval_subset)):
        row = eval_subset.iloc[i]
        enc = tokenizer(row["input_text"], return_tensors="pt",
                        truncation=True, max_length=MAX_INPUT_LEN)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        enc_hidden = model.backbone.encoder(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            return_dict=True,
        ).last_hidden_state
        mask   = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (enc_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        pred   = model.reg_head(pooled).cpu().numpy()
        all_preds.append(pred[0])

preds_orig = scaler.inverse_transform(np.array(all_preds))
true_orig  = eval_subset[NUTRITION_COLS].to_numpy(dtype=np.float32)

print("Mean Absolute Error per nutrition column (on 200 val samples):")
print("-" * 40)
for i, col in enumerate(NUTRITION_COLS):
    mae = mean_absolute_error(true_orig[:, i], preds_orig[:, i])
    print(f"  {col:<25}: {mae:.2f}")